## 09 Human In the Loop (HTIL)

It is often the case that agents will need human intervention to resolve certain issues. This can be achieved via Human-in-the-loop (or HTIL). In LangChain agents, when defining an agent, you can specify for which tools calls you'll need human feedback. When one of those tools is called, an _interrupt_ is raised, asking for a human response. You can setup various _allowed_ responses, such as approvals, rejections and edits.



In [11]:
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from dataclasses import dataclass

import langchain
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase

print(f"Using langchain version: {langchain.__version__}")

load_dotenv(override=True)
console = Console()

Using langchain version: 1.2.14


**Step 1:**

As a first step, let's load the `Chinook` database and it's schema. The schema will help our agent generate the query faster.  

Next we define the runtime context for the LLM to hold the database connection as well as the schema information so that the agent is able to minimize loops before generating text-to-SQL. 

We also define our `execute_sql` tool and the system prompt - this time, we follow the simple modification of system prompt rather than use dynamic prompting with the `@dynamic_prompt` that we used in the [previous notebook](08_middleware.ipynb).

In [12]:
from langchain_community.utilities import SQLDatabase
from typing import TypedDict, Optional, List
from pydantic import BaseModel

from langchain.tools import tool
from langgraph.runtime import get_runtime

# connect to our database -> in path db/chinook.db
db = SQLDatabase.from_uri("sqlite:///db/chinook.db")
schema = db.get_table_info()


# define our runtime context - we'll pass in the schema this time
class RuntimeContext(TypedDict):
    db: SQLDatabase
    db_schema: str


# define our tool to execute sql
@tool
def execute_sql(query: str) -> str:
    """execute query provided by user
    Args:
        query (str): SQL query to execute
    Returns:
        str: result of the query execution or error message
    """
    # get instance of db in context
    db: SQLDatabase = get_runtime().context["db"]
    try:
        result = db.run(query)
    except Exception as e:
        return f"Error occurred while executing SQL query: {e}"
    return str(result)


# define our system prompt with schema injected
SYSTEM_PROMPT = (
    """You are a careful SQLite Analyst.

Rules:
- Always think step-by-step
- When you need data, call the tool 'execute_sql' with ONE select query
- Read-only only; NO INSERT/UPDATE/DELETE/DROP/CREATE/REPLACE/TRUNCATE/ALTER
- Limit to 5 rows at the output, unless the user explicitly asks for more
- If the tool returns "Error:", revise the SQL and try again
- Prefer explicit column list, avoid SELECT *
- Here is the database schema you can refer to to generate the SQL
{database_schema}
"""
).format(database_schema=schema)

**Step 2**

HTIL are interrupts you define on certain tools that the agent has access to - you can choose which tools need the HTIL interrupt. This interrupt is "fired" AFTER the tool call, but BEFORE the LLM sees output of the tool. The internal Agent loop is interrupted at this point, the graph state is saved in LangGraph memory and control shifts to the caller of the agent (i.e. the Human). When we define the HTIL for an Agent, we specify what actions the Human is allowed to take - it can be one of ["accept", "reject", "edit"]

Next we create our agent where we specify our HTIL interrupt as follows:

In [13]:
# define our agent
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    context_schema=RuntimeContext,
    middleware=[
        HumanInTheLoopMiddleware(
            # interrup the flow AFTER return from execute_sql, but BEFORE the LLM
            # gets to see it. User is allowed to either "approve" or "reject"
            # there is another possible option "edit", which we have not used here!
            interrupt_on={"execute_sql": {"allowed_decisions": ["approve", "reject"]}},
        ),
        # define additional HTIL middleware instance if you have more tools
    ],
)

**Step 3**

Here is how we call the agent - 

In [14]:
from langgraph.types import Command
from rich.console import Console

console = Console()

question = "What are the names (first & last) of all employees?"
config = {"configurable": {"thread_id": "1024"}}

response = agent.invoke(
    {"messages": {"role": "user", "content": question}},
    config=config,
    context=RuntimeContext(db=db, db_schema=schema),
)

if "__interrupt__" in response:
    description = response["__interrupt__"][-1].value["action_requests"][-1][
        "description"
    ]
    console.print(f"[bright_red]{80*"="}[/bright_red]")
    console.print(f"[bright_red]Interrupt: {description}[/bright_red]")
    result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "reject", "message": "database is offline"}]
            },
        ),
        config=config,
        context=RuntimeContext(db=db, db_schema=schema),
    )
    console.print(f"[red]{80*"="}[/red]")

for msg in result["messages"]:
    msg.pretty_print()

================================================================================

Interrupt: Tool execution requires approval

Tool: execute_sql
Args: {'query': 'SELECT FirstName, LastName FROM employees ORDER BY EmployeeId LIMIT 5;'}

================================================================================

================================ Human Message =================================

What are the names (first & last) of all employees?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_R0d1yLF1pBkWJJGA0VSA7YJA)
 Call ID: call_R0d1yLF1pBkWJJGA0VSA7YJA
  Args:
    query: SELECT FirstName, LastName FROM employees ORDER BY EmployeeId LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

database is offline
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_FWiyyx7sCzEhuqGQjxQH6SzU)
 Call ID: call_FWiyyx7sCzEhuqGQjxQH6SzU
  Args:
    query: SELECT FirstName, LastName FROM employees ORDER BY EmployeeId LIMIT 5;


In [15]:
config = {"configurable": {"thread_id": "2"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db, db_schema=schema),
)

while "__interrupt__" in result:
    description = result["__interrupt__"][-1].value["action_requests"][-1][
        "description"
    ]
    print(f"\033[1;3;31m{80 * '-'}\033[0m")
    print(f"\033[1;3;31m Interrupt:{description}\033[0m")

    result = agent.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )

for msg in result["messages"]:
    msg.pretty_print()

--------------------------------------------------------------------------------
 Interrupt:Tool execution requires approval

Tool: execute_sql
Args: {'query': 'SELECT FirstName, LastName FROM employees LIMIT 5;'}
================================ Human Message =================================

What are the names (first & last) of all employees?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_dxnnMLjicpkk507gKOdxpt68)
 Call ID: call_dxnnMLjicpkk507gKOdxpt68
  Args:
    query: SELECT FirstName, LastName FROM employees LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Andrew', 'Adams'), ('Nancy', 'Edwards'), ('Jane', 'Peacock'), ('Margaret', 'Park'), ('Steve', 'Johnson')]
================================== Ai Message ==================================

Here are the first 5 employees (first & last names):

- Andrew Adams
- Nancy Edwards
- Jane Peacock
- Margaret P